# MediSense-Cardio -- Degradation Harness (Pilot)

**What this notebook does:** the proof-of-concept demo (`MediSense_Cardio_Demo_v2.ipynb`) showed the pipeline
works on 6 clean, hand-picked recordings. This notebook is the next real step: a **stratified sample** drawn
properly (not hand-picked), each recording rendered **clean and then deliberately blurred at several levels**,
and the digitizer's accuracy measured at each level -- plus, critically, **how long each run actually takes**,
which determines whether the full ~500-record x 6-axis study is a same-day job or needs to run over days.

This is a **pilot**: a small sample (10 records) x **one** degradation axis (blur), end to end. Once this runs
clean and the timing is known, the same code extends to the full sample and all 6 axes from the plan.

Run all cells top to bottom (**Runtime -> Run all**). Uses a GPU runtime if available (Runtime -> Change
runtime type -> T4 GPU) -- the digitizer auto-detects and uses it, which should be much faster than the local
CPU-only runs this project did earlier.


## Step 0 -- Install the base tools


In [ ]:
!pip install -q wfdb neurokit2 opencv-python-headless pandas numpy scipy matplotlib
print("Base tools installed.")


## Step 1 -- Draw the stratified sample

PTB-XL labels every recording with SCP diagnostic codes; `scp_statements.csv` maps each code to one of five
superclasses (NORM, MI, STTC, CD, HYP). A recording is **normal** if its highest-confidence code's superclass
is NORM, **abnormal** otherwise. Per the project plan, the sample needs a real fraction of abnormal cases --
without them, false reassurance can't be observed at all -- so this is computed and reported explicitly, not
assumed.


In [ ]:
import pandas as pd, ast, ssl, urllib.request

# PhysioNet serves the open PTB-XL files directly over HTTPS -- fetching just the two
# small label CSVs (not the ~1.8GB archive) is all that's needed to pick the sample.
BASE = 'https://physionet.org/files/ptb-xl/1.0.3/'
ctx = ssl.create_default_context()

def fetch_csv(name):
    with urllib.request.urlopen(BASE + name, context=ctx) as r:
        with open(name, 'wb') as f:
            f.write(r.read())
    return pd.read_csv(name)

database = fetch_csv('ptbxl_database.csv')
scp_statements = fetch_csv('scp_statements.csv').rename(columns={'Unnamed: 0': 'code'})
code_to_class = dict(zip(scp_statements['code'], scp_statements['diagnostic_class']))

def dominant_class(scp_codes_str):
    codes = ast.literal_eval(scp_codes_str)  # e.g. "{'NORM': 100.0, 'LVOLT': 0.0}"
    if not codes:
        return None
    top_code = max(codes, key=codes.get)
    return code_to_class.get(top_code)

database['diagnostic_class'] = database['scp_codes'].apply(dominant_class)
database['is_abnormal'] = database['diagnostic_class'].notna() & (database['diagnostic_class'] != 'NORM')

n_total = len(database)
n_abnormal = int(database['is_abnormal'].sum())
print(f"PTB-XL: {n_total} records total, {n_abnormal} ({100*n_abnormal/n_total:.1f}%) classified abnormal.")


In [ ]:
# Pilot sample: 10 records, roughly half abnormal -- enough to sanity-check the
# harness end-to-end before committing to the full ~500-record draw.
PILOT_N = 10
rng_seed = 42

normal_pool = database[~database['is_abnormal']]
abnormal_pool = database[database['is_abnormal']]

pilot = pd.concat([
    normal_pool.sample(n=PILOT_N // 2, random_state=rng_seed),
    abnormal_pool.sample(n=PILOT_N - PILOT_N // 2, random_state=rng_seed),
]).reset_index(drop=True)

print(f"Pilot sample: {len(pilot)} records -- {int(pilot['is_abnormal'].sum())} abnormal, {int((~pilot['is_abnormal']).sum())} normal.")
pilot[['ecg_id', 'filename_lr', 'diagnostic_class', 'is_abnormal']]


## Step 2 -- Fetch the sampled recordings

Each record is streamed directly from PhysioNet via `wfdb` -- no need to download the full archive for a
10-record pilot (the full-scale run will do the same, just for ~500 records instead of 10).


In [ ]:
import wfdb, os

# LOCAL PATCH (same as the demo notebook): Colab's numpy removed the old
# np.fromstring binary-read path that wfdb's own code still calls internally.
import numpy as np
_original_fromstring = np.fromstring
def _patched_fromstring(string, dtype=float, count=-1, sep=''):
    if sep == '':
        return np.frombuffer(string, dtype=dtype, count=count)
    return _original_fromstring(string, dtype=dtype, count=count, sep=sep)
np.fromstring = _patched_fromstring

os.makedirs('/content/data/pilot', exist_ok=True)
record_ids = []
for _, row in pilot.iterrows():
    # filename_lr looks like 'records100/00000/00001_lr' -- pn_dir is everything
    # up to the record name itself.
    rel_dir = os.path.dirname(row['filename_lr'])
    rid = os.path.basename(row['filename_lr'])
    record = wfdb.rdrecord(rid, pn_dir=f'ptb-xl/1.0.3/{rel_dir}')
    wfdb.wrsamp(rid, fs=record.fs, units=record.units, sig_name=record.sig_name,
                p_signal=record.p_signal, write_dir='/content/data/pilot')
    record_ids.append(rid)
    print(f"{rid}: fetched -- abnormal={row['is_abnormal']}, class={row['diagnostic_class']}")


## Step 3 -- Draw them as realistic ECG pictures (clean baseline)


In [ ]:
!git clone --depth 1 https://github.com/alphanumericslab/ecg-image-kit.git /content/ecg-image-kit
%cd /content/ecg-image-kit/codes/ecg-image-generator

# LOCAL PATCHES -- identical to the proven demo notebook (see its Step 2 markdown
# for the full explanation of each). Reused verbatim, not re-derived.
from pathlib import Path

HW_IMPORT = "from HandwrittenText.generate import get_handwritten"
p = Path('gen_ecg_image_from_data.py'); t = p.read_text()
if ("                " + HW_IMPORT) not in t:
    t = t.replace(HW_IMPORT + "\n", "")
    t = t.replace("            if(hw_text):", "            if(hw_text):\n                " + HW_IMPORT)
    p.write_text(t)

p = Path('helper_functions.py'); t = p.read_text()
if "from imgaug import augmenters as iaa\n" in t:
    p.write_text(t.replace("from imgaug import augmenters as iaa\n", ""))

p = Path('CreasesWrinkles/creases.py'); t = p.read_text()
if "import imgaug" in t:
    t = t.replace("import imgaug as ia\n", "").replace("from imgaug import augmenters as iaa\n", "")
    p.write_text(t)

p = Path('ImageAugmentation/augment.py'); t = p.read_text()
if "import imgaug" in t and "def get_augment" in t:
    lines = t.split('\n')
    top_imports = [l for l in lines if 'imgaug' in l and not l.strip().startswith('#')]
    for imp in top_imports:
        if lines.count(imp):
            lines.remove(imp)
    for i, l in enumerate(lines):
        if l.strip().startswith('def get_augment'):
            indent = '    '
            for imp in top_imports:
                lines.insert(i + 1, indent + imp.strip())
            break
    p.write_text('\n'.join(lines))
print("ecg-image-kit patched.")


In [ ]:
requirements_core = '''imageio
imutils
pillow
scikit-image
scikit-learn
scipy
matplotlib
joblib
numpy
pandas
wfdb
pyyaml
qrcode
'''
with open('requirements-core.txt', 'w') as f:
    f.write(requirements_core)
!pip install -q -r requirements-core.txt
print("Image-drawing tool ready.")


In [ ]:
!python gen_ecg_images_from_data_batch.py -i /content/data/pilot -o /content/images_clean --print_header
print("\nClean images generated:")
!ls /content/images_clean/*.png


## Step 4 -- Degrade: apply blur at several levels

This axis isn't one of ecg-image-kit's built-in flags, so it's a direct OpenCV post-process on the rendered
clean image -- the same approach the plan uses for JPEG compression, brightness, and grid contrast later.
5 levels + the clean baseline, matching the project plan's degradation-axis design.


In [ ]:
import cv2, glob, shutil

# Kernel size controls blur strength; must be odd. 0 = clean (no blur applied).
BLUR_LEVELS = [0, 3, 7, 11, 15, 21]

clean_images = sorted(glob.glob('/content/images_clean/*.png'))
os.makedirs('/content/images_degraded', exist_ok=True)

variant_map = []  # (record_id, level, output_path)
for img_path in clean_images:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path)
    for level in BLUR_LEVELS:
        out_path = f'/content/images_degraded/{stem}__blur{level}.png'
        if level == 0:
            shutil.copy(img_path, out_path)
        else:
            blurred = cv2.GaussianBlur(img, (level, level), 0)
            cv2.imwrite(out_path, blurred)
        variant_map.append({'stem': stem, 'blur_level': level, 'image_path': out_path})

variant_df = pd.DataFrame(variant_map)
print(f"{len(variant_df)} image variants ready ({len(clean_images)} records x {len(BLUR_LEVELS)} blur levels).")


## Step 5 -- Set up the digitizer


In [ ]:
!git clone https://github.com/Ahus-AIM/Open-ECG-Digitizer.git /content/Open-ECG-Digitizer
%cd /content/Open-ECG-Digitizer
!git lfs install
!git lfs pull

path = 'src/utils.py'
with open(path) as f:
    content = f.read()
content = content.replace(
    "from ray.tune import Stopper",
    "try:\n    from ray.tune import Stopper\nexcept ImportError:\n    Stopper = object  # ray not needed for inference"
)
with open(path, 'w') as f:
    f.write(content)
print("Patched: ray import made optional")


In [ ]:
!pip install -q torch torchvision yacs scikit-image tqdm torch-tps
print("Digitizer tools ready.")


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

config_content = f'''MODEL:
  class_path: 'src.model.inference_wrapper.InferenceWrapper'
  KWARGS:
    config:
      SIGNAL_EXTRACTOR:
        class_path: 'src.model.signal_extractor.SignalExtractor'
        KWARGS: {{}}
      PERSPECTIVE_DETECTOR:
        class_path: 'src.model.perspective_detector.PerspectiveDetector'
        KWARGS:
          num_thetas: 250
      DEWARPER:
        class_path: 'src.model.dewarper.Dewarper'
        KWARGS:
          abs_peak_threshold: 0.1
      SEGMENTATION_MODEL:
        class_path: 'src.model.unet.UNet'
        weight_path: './weights/unet_weights_07072025.pt'
        KWARGS:
          num_in_channels: 3
          num_out_channels: 4
          dims: [32, 64, 128, 256, 320, 320, 320, 320]
          depth: 2
      CROPPER:
        class_path: 'src.model.cropper.Cropper'
        KWARGS:
          granularity: 80
          percentiles: [0.02, 0.98]
          alpha: 0.85
      PIXEL_SIZE_FINDER:
        class_path: 'src.model.pixel_size_finder.PixelSizeFinder'
        KWARGS:
          min_number_of_grid_lines: 30
          max_number_of_grid_lines: 70
          lower_grid_line_factor: 0.5
      LAYOUT_IDENTIFIER:
        class_path: 'src.model.lead_identifier.LeadIdentifier'
        config_path: 'src/config/lead_layouts_george-moody-2024.yml'
        unet_config_path: 'src/config/lead_name_unet.yml'
        unet_weight_path: './weights/lead_name_unet_weights_07072025.pt'
        KWARGS:
          debug: false
          device: '{device}'
          possibly_flipped: false
          target_num_samples: 10000
          required_valid_samples: 2
    device: '{device}'
    resample_size: 3000
    rotate_on_resample: true
    enable_timing: false
    apply_dewarping: false
DATA:
  images_path: '/content/images_degraded'
  image_extensions: ['.png', '.jpg', '.jpeg', '.JPG']
  save_mode: 'all'
  layout_should_include_substring: false
  output_path: '/content/digitized'
  clear_output_dir_if_exists: true
'''
with open('/content/Open-ECG-Digitizer/pilot_config.yml', 'w') as f:
    f.write(config_content)
print("Config written.")


## Step 6 -- Run the digitizer on every variant, and time it

This is the measurement the pilot exists to produce: real wall-clock time for this environment (GPU or CPU),
which turns the full-scale sweep's runtime from a guess into a number.


In [ ]:
import time
%cd /content/Open-ECG-Digitizer
t0 = time.time()
!python -m src.digitize --config /content/Open-ECG-Digitizer/pilot_config.yml
elapsed = time.time() - t0
n_variants = len(variant_df)
print(f"\nDigitized {n_variants} image variants in {elapsed:.1f}s -- {elapsed/n_variants:.2f}s/image average.")

FULL_SWEEP_VARIANTS = 500 * 31  # ~500 records x (6 axes x 5 levels + 1 clean baseline)
est_hours = (elapsed / n_variants) * FULL_SWEEP_VARIANTS / 3600
print(f"At this rate, the full ~500-record x 31-variant sweep would take an estimated {est_hours:.1f} hours.")


## Step 7 -- Compare every variant against ground truth

For each record, at each blur level: heart-rate error (T2) and signal-to-noise ratio (T1) against the true
recording. This is the false-reassurance study's T1/T2 output, on the pilot scale.


In [ ]:
import neurokit2 as nk

FS_CANONICAL = 1000.0
results = []

for _, variant in variant_df.iterrows():
    stem, level = variant['stem'], variant['blur_level']
    gt_rid = stem.split('-')[0]  # ecg-image-kit appends '-0' etc per rendered page
    record = wfdb.rdrecord(f'/content/data/pilot/{gt_rid}')
    gt_signal = record.p_signal[:, record.sig_name.index('II')]
    fs_gt = record.fs

    csv_path = f'/content/digitized/{stem}__blur{level}_timeseries_canonical.csv'
    if not os.path.exists(csv_path):
        results.append({'record': gt_rid, 'blur_level': level, 'dig_hr': None, 'diff_bpm': None, 'snr_db': None})
        continue

    digitized = pd.read_csv(csv_path)
    dig_signal = digitized['II'].dropna().to_numpy() / 1000.0

    gt_peaks, _ = nk.ecg_peaks(gt_signal, sampling_rate=fs_gt)
    gt_hr = np.nanmean(nk.ecg_rate(gt_peaks, sampling_rate=fs_gt, desired_length=len(gt_signal)))

    try:
        dig_peaks, _ = nk.ecg_peaks(dig_signal, sampling_rate=FS_CANONICAL)
        dig_hr = np.nanmean(nk.ecg_rate(dig_peaks, sampling_rate=FS_CANONICAL, desired_length=len(dig_signal)))
    except Exception:
        dig_hr = float('nan')

    diff = abs(gt_hr - dig_hr) if not np.isnan(dig_hr) else None

    # T1: crude SNR proxy -- resample digitized to GT length, treat GT as signal,
    # difference as noise. A real implementation would time-align via cross-correlation
    # first; this pilot version flags that as a known simplification, not a final metric.
    snr_db = None
    if len(dig_signal) > 10:
        n = min(len(gt_signal), len(dig_signal))
        noise = gt_signal[:n] - dig_signal[:n] * (gt_signal[:n].std() / (dig_signal[:n].std() + 1e-9))
        signal_power = np.mean(gt_signal[:n] ** 2)
        noise_power = np.mean(noise ** 2)
        if noise_power > 0:
            snr_db = float(10 * np.log10(signal_power / noise_power))

    results.append({
        'record': gt_rid, 'blur_level': level,
        'gt_hr': round(gt_hr, 1), 'dig_hr': round(dig_hr, 1) if not np.isnan(dig_hr) else None,
        'diff_bpm': round(diff, 2) if diff is not None else None,
        'snr_db': round(snr_db, 1) if snr_db is not None else None,
    })

results_df = pd.DataFrame(results)
results_df.to_csv('/content/pilot_results.csv', index=False)
results_df


### Error vs. blur level -- does the trend look right?


In [ ]:
summary = results_df.groupby('blur_level')['diff_bpm'].agg(['mean', 'count'])
print("Average heart-rate error by blur level (higher level = more blur):")
summary


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for rid, grp in results_df.groupby('record'):
    ax.plot(grp['blur_level'], grp['diff_bpm'], marker='o', label=rid, alpha=0.6)
ax.set_xlabel('Blur kernel size (0 = clean)')
ax.set_ylabel('Heart-rate error vs. ground truth (bpm)')
ax.set_title('Pilot: does heart-rate error rise as blur increases?')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Summary

This pilot proves the **full harness loop** -- stratified sample -> fetch -> render -> degrade -> digitize ->
compare -- runs correctly end to end, not just the clean-image case the earlier demo covered. Two things this
notebook's output answers directly:

1. **Does error rise with degradation?** (see the plot above) -- the core hypothesis the whole study is built on.
2. **How long does the full study actually take?** (see Step 6's printed estimate) -- turns the full-scale
   sweep from a guess into a planned, budgeted run.

**Known simplifications in this pilot, to fix before the full run:**
- SNR (T1) uses a crude same-length overlay, not a real time-aligned comparison -- fine for a pilot sanity
  check, not for the paper's actual T1 numbers.
- Only one degradation axis (blur) is exercised here; the plan's other five (rotation, JPEG, resolution,
  brightness, grid contrast) reuse this same render/degrade/digitize/compare structure.
- T3 (the explanation-correctness taxonomy -- false reassurance, false alarm, value error, silent drop) isn't
  computed yet -- it needs PR/QRS/QT extraction and the reference-range thresholds, not just heart rate.
